# RFM Customer Segmentation Analysis
**Dataset:** Online Retail II (UCI Machine Learning Repository)  
**Period:** 2010–2011 | **Market:** UK E-commerce  

This notebook walks through the full RFM (Recency, Frequency, Monetary) pipeline:
1. Load & explore raw transactional data
2. Clean and preprocess
3. Compute RFM metrics
4. Score and segment customers
5. Visualise segments
6. Export for dashboard

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("All libraries loaded!")

## 2. Load Data

In [ ]:
df = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2010-2011')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)

**Observations:**
- 541,910 rows across 8 columns
- `Customer ID` has 135,080 missing values — these rows cannot be attributed to any customer and must be dropped
- `Description` has 1,454 missing values — minor, not critical for RFM
- `InvoiceDate` is already parsed as `datetime64`

## 3. Data Cleaning

In [ ]:
# Step 1: Remove rows where Customer ID is missing
df = df.dropna(subset=['Customer ID'])

# Step 2: Convert Customer ID to integer then string
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# Step 3: Remove cancelled orders (Invoice starts with 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Step 4: Keep only positive quantities and prices
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

print("Clean data shape:", df.shape)

After cleaning: **397,885 rows** — removed ~27% of records (nulls, cancellations, invalid entries).

## 4. Compute RFM Metrics

In [ ]:
# Set the reference date (day after last transaction in dataset)
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Create a new column: revenue per row
df['TotalPrice'] = df['Quantity'] * df['Price']

# Calculate R, F, M for each customer
rfm = df.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary  = ('TotalPrice', 'sum')
).reset_index()

print(rfm.shape)
rfm.head(10)

In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe()

## 5. RFM Scoring (1–5 scale)

In [ ]:
# For Recency: LOWER days = BETTER = score 5
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])

# For Frequency: HIGHER = BETTER = score 5
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# For Monetary: HIGHER = BETTER = score 5
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# Combine into one RFM score string e.g. "555" = best customer
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print(rfm[['Customer ID', 'Recency', 'Frequency', 'Monetary', 'RFM_Score']].head(10))

## 6. Customer Segmentation

In [ ]:
def assign_segment(score):
    r = int(score[0])
    f = int(score[1])
    m = int(score[2])

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal Customer'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and f <= 2 and m <= 2:
        return 'Promising'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Cannot Lose Them'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

print(rfm['Segment'].value_counts())

## 7. Visualisations

In [ ]:
seg_colors = {
    'Champion': '#4ade80',
    'Loyal Customer': '#60a5fa',
    'New Customer': '#a78bfa',
    'Promising': '#fbbf24',
    'At Risk': '#f87171',
    'Cannot Lose Them': '#f472b6',
    'Lost': '#6b7280'
}

# Segment distribution — pie chart
seg_count = rfm['Segment'].value_counts().reset_index()
seg_count.columns = ['Segment', 'Count']

fig = px.pie(seg_count, names='Segment', values='Count',
             color='Segment', color_discrete_map=seg_colors,
             hole=0.45, title='Customer Segment Distribution')
fig.show()

In [ ]:
# Revenue by segment — horizontal bar chart
seg_rev = rfm.groupby('Segment')['Monetary'].sum().reset_index()
seg_rev.columns = ['Segment', 'Revenue']
seg_rev = seg_rev.sort_values('Revenue', ascending=True)

fig2 = px.bar(seg_rev, x='Revenue', y='Segment', orientation='h',
              color='Segment', color_discrete_map=seg_colors,
              title='Total Revenue by Segment')
fig2.show()

In [ ]:
# Recency vs Monetary scatter plot
fig3 = px.scatter(rfm, x='Recency', y='Monetary', color='Segment',
                  color_discrete_map=seg_colors, opacity=0.6,
                  hover_data=['Customer ID', 'Frequency'],
                  title='Recency vs Monetary Value by Segment')
fig3.show()

In [ ]:
# RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rfm['Recency'].hist(ax=axes[0], bins=30, color='#60a5fa', edgecolor='white')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days since last purchase')

rfm['Frequency'].hist(ax=axes[1], bins=30, color='#4ade80', edgecolor='white')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of orders')

rfm['Monetary'].hist(ax=axes[2], bins=30, color='#fbbf24', edgecolor='white')
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total spend (£)')

plt.tight_layout()
plt.show()

## 8. Export Results

In [ ]:
rfm.to_csv('../data/rfm_segments.csv', index=False)
print("Saved! rfm_segments.csv is in your data folder.")
print(f"\nFinal dataset: {rfm.shape[0]} customers across {rfm['Segment'].nunique()} segments")
rfm.head()